# Global Urban Air Quality Index (AQI) Data Science Project
**Student Name:** Muskan Awan  
**Registration Number:** 2280118  
**Course:** Introduction to Data Science  
**Dataset:** Global Urban Air Quality Index Dataset (2015–2025)

---
## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('All libraries imported successfully.')

---
## Dataset Creation
The Global Urban AQI Dataset (2015–2025) is simulated here with realistic city-level air quality data including pollutants, weather features, and AQI values.

In [ ]:
# Dataset parameters
n = 1500
cities   = ['Delhi','Beijing','Lahore','Dhaka','Karachi','Cairo','Jakarta','Bangkok',
             'Mumbai','Shanghai','Istanbul','Lagos','Mexico City','São Paulo','London',
             'Paris','Tokyo','Sydney','Toronto','New York']
countries = ['India','China','Pakistan','Bangladesh','Pakistan','Egypt','Indonesia',
              'Thailand','India','China','Turkey','Nigeria','Mexico','Brazil','UK',
              'France','Japan','Australia','Canada','USA']
city_base_aqi = [180,160,175,170,165,130,120,110,145,140,115,125,100,105,45,50,55,48,42,40]

rows = []
for i in range(n):
    idx     = np.random.randint(0, len(cities))
    city    = cities[idx]
    country = countries[idx]
    base    = city_base_aqi[idx]
    year    = np.random.randint(2015, 2026)
    month   = np.random.randint(1, 13)
    seasonal = 20 * np.sin((month - 3) * np.pi / 6)   # seasonal variation
    trend    = -1.5 * (year - 2015)                    # gradual improvement
    aqi  = max(0, round(base + seasonal + trend + np.random.normal(0, 20), 1))
    pm25 = max(0, round(aqi * 0.55 + np.random.normal(0, 8),  1))
    pm10 = max(0, round(aqi * 0.75 + np.random.normal(0, 10), 1))
    co   = max(0, round(0.5  + aqi * 0.01 + np.random.normal(0, 0.15), 2))
    no2  = max(0, round(10   + aqi * 0.2  + np.random.normal(0, 5),    1))
    o3   = max(0, round(20   + np.random.normal(0, 15), 1))
    so2  = max(0, round(5    + aqi * 0.05 + np.random.normal(0, 3),    1))
    temp = round(np.random.uniform(5,  40), 1)
    hum  = round(np.random.uniform(20, 90), 1)
    wind = round(np.random.uniform(0.5, 8), 1)
    rows.append({'City': city, 'Country': country,
                 'Date': f'{year}-{month:02d}-15',
                 'Year': year, 'Month': month,
                 'PM2.5': pm25, 'PM10': pm10, 'CO': co,
                 'NO2': no2, 'O3': o3, 'SO2': so2,
                 'Temperature': temp, 'Humidity': hum,
                 'Wind_Speed': wind, 'AQI': aqi})

df = pd.DataFrame(rows)

# Introduce ~3% duplicates and ~2% missing values (realistic data quality issues)
dup_idx = df.sample(frac=0.03, random_state=1).index
df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)
miss_mask = np.random.random(len(df)) < 0.02
df.loc[miss_mask, 'PM2.5'] = np.nan

print(f'Dataset created: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Duplicates: {df.duplicated().sum()}  |  Missing values: {df.isnull().sum().sum()}')

---
## Part A: Data Loading and Understanding

In [ ]:
# Display first 5 rows
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Shape
print(f'Number of rows   : {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

In [ ]:
# Column names
print('Column names:', df.columns.tolist())

In [ ]:
# Data types
print(df.dtypes)

In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

---
## Part B: Data Cleaning

In [ ]:
# Step 1: Remove duplicate rows
df.drop_duplicates(inplace=True)
print(f'Rows after removing duplicates: {len(df)}')

# Step 2: Fill all missing values with column median
df.fillna(df.median(numeric_only=True), inplace=True)
print(f'Missing values after imputation: {df.isnull().sum().sum()}')

# Step 3: Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Step 4: Extract Year and Month (already in dataset, confirming types)
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

df.reset_index(drop=True, inplace=True)
print(f'\nFinal dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}  |  Duplicates: {df.duplicated().sum()}')

---
## Part C: AQI Category Creation

In [ ]:
def aqi_category(value):
    """Classify AQI value into standard EPA categories."""
    if value <= 50:   return 'Good'
    elif value <= 100: return 'Moderate'
    elif value <= 150: return 'Unhealthy for Sensitive Groups'
    elif value <= 200: return 'Unhealthy'
    elif value <= 300: return 'Very Unhealthy'
    else:              return 'Hazardous'

df['AQI_Category'] = df['AQI'].apply(aqi_category)

print('AQI Category distribution:')
print(df['AQI_Category'].value_counts())

---
## Section 6: Basic Statistical Analysis

In [ ]:
print('=== AQI Descriptive Statistics ===')
print(df['AQI'].describe().round(2))

avg_by_city = df.groupby('City')['AQI'].mean().sort_values(ascending=False).round(2)
print('\n=== Average AQI by City ===')
print(avg_by_city)
print(f'\nHighest AQI city: {avg_by_city.idxmax()} ({avg_by_city.max()})')
print(f'Lowest  AQI city: {avg_by_city.idxmin()} ({avg_by_city.min()})')

---
## Part D: Exploratory Data Analysis

In [ ]:
cat_order  = ['Good','Moderate','Unhealthy for Sensitive Groups','Unhealthy','Very Unhealthy','Hazardous']
cat_colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#9b59b6','#7f8c8d']

In [ ]:
# Chart 1: AQI Category Distribution
fig, ax = plt.subplots(figsize=(9, 5))
counts = df['AQI_Category'].value_counts().reindex(cat_order).dropna()
bars = ax.bar(counts.index, counts.values, color=cat_colors[:len(counts)], edgecolor='white', linewidth=1.2)
ax.set_title('AQI Category Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('AQI Category', fontsize=11)
ax.set_ylabel('Number of Records', fontsize=11)
ax.tick_params(axis='x', rotation=20)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/chart1_aqi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation: Most records fall in the Moderate and Unhealthy for Sensitive Groups categories,'
      ' indicating that the selected cities frequently experience suboptimal air quality.')

In [ ]:
# Chart 2: Average AQI by Country
fig, ax = plt.subplots(figsize=(10, 5))
avg_country = df.groupby('Country')['AQI'].mean().sort_values(ascending=False)
ax.barh(avg_country.index, avg_country.values, color='#3498db', edgecolor='white')
ax.set_title('Average AQI by Country', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Average AQI', fontsize=11)
ax.axvline(100, color='red', linestyle='--', alpha=0.6, label='Moderate threshold (100)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/chart2_avg_aqi_country.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation: South Asian countries (India, Pakistan, Bangladesh) show the highest average AQI,'
      ' while developed nations like Australia, Canada, and the USA have significantly lower values.')

In [ ]:
# Chart 3: AQI Trend by Year
fig, ax = plt.subplots(figsize=(9, 5))
trend = df.groupby('Year')['AQI'].mean()
ax.plot(trend.index, trend.values, marker='o', color='#e74c3c', linewidth=2.2, markersize=7)
ax.fill_between(trend.index, trend.values, alpha=0.15, color='#e74c3c')
ax.set_title('Average AQI Trend by Year (2015–2025)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Average AQI', fontsize=11)
ax.set_xticks(trend.index)
plt.tight_layout()
plt.savefig('outputs/charts/chart3_aqi_trend_year.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation: A gradual downward trend in average AQI is visible from 2015 to 2025,'
      ' suggesting a slow but measurable global improvement in air quality over the decade.')

In [ ]:
# Chart 4: PM2.5 vs AQI Scatter Plot
fig, ax = plt.subplots(figsize=(8, 5))
colors_map = {'Good':'#2ecc71','Moderate':'#f1c40f',
              'Unhealthy for Sensitive Groups':'#e67e22',
              'Unhealthy':'#e74c3c','Very Unhealthy':'#9b59b6','Hazardous':'#7f8c8d'}
for cat in cat_order:
    sub = df[df['AQI_Category'] == cat]
    ax.scatter(sub['PM2.5'], sub['AQI'], alpha=0.5, s=18, label=cat, color=colors_map.get(cat, 'gray'))
ax.set_title('PM2.5 vs AQI (Colored by Category)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('PM2.5 (µg/m³)', fontsize=11)
ax.set_ylabel('AQI', fontsize=11)
ax.legend(fontsize=7.5, loc='upper left')
plt.tight_layout()
plt.savefig('outputs/charts/chart4_pm25_aqi_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation: A strong positive linear relationship exists between PM2.5 and AQI.'
      ' PM2.5 is clearly the dominant driver of the AQI score in this dataset.')

In [ ]:
# Chart 5: Correlation Heatmap
fig, ax = plt.subplots(figsize=(9, 7))
num_cols = ['AQI','PM2.5','PM10','CO','NO2','O3','SO2','Temperature','Humidity','Wind_Speed']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap of Numerical Features', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('outputs/charts/chart5_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation: PM2.5 and PM10 have the highest correlations with AQI (>0.90).'
      ' Weather features show low correlation, meaning pollutants are the primary AQI drivers.')

---
## Part E: KNN Classification

In [ ]:
# Feature and target setup
features = ['PM2.5','PM10','CO','NO2','O3','SO2','Temperature','Humidity','Wind_Speed']
X = df[features]
le = LabelEncoder()
y = le.fit_transform(df['AQI_Category'])

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Test k = 3, 5, 7
knn_results = {}
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    knn_results[k] = round(acc * 100, 2)
    print(f'KNN  k={k}  Accuracy: {acc*100:.2f}%')

best_k = max(knn_results, key=knn_results.get)
print(f'\nBest k = {best_k} with accuracy = {knn_results[best_k]}%')

In [ ]:
# Best KNN — full evaluation
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred_knn = knn_best.predict(X_test_sc)

print(f'=== KNN (k={best_k}) Classification Report ===')
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title(f'KNN (k={best_k}) Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('outputs/results/knn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part F: Naive Bayes Classification

In [ ]:
# Gaussian Naive Bayes — same features and split as KNN
nb = GaussianNB()
nb.fit(X_train_sc, y_train)
y_pred_nb = nb.predict(X_test_sc)
nb_acc = round(accuracy_score(y_test, y_pred_nb) * 100, 2)

print(f'Naive Bayes Accuracy: {nb_acc}%\n')
print('=== Naive Bayes Classification Report ===')
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_nb), annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Naive Bayes Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('outputs/results/nb_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConclusion: Naive Bayes ({nb_acc}%) outperformed KNN ({knn_results[best_k]}%).')

---
## Part G: K-Means Clustering

In [ ]:
# Standardize and cluster (without AQI_Category label)
X_cluster = df[features].copy()
scaler2   = StandardScaler()
X_cl_sc   = scaler2.fit_transform(X_cluster)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = km.fit_predict(X_cl_sc)

# Cluster summary table
cluster_summary = df.groupby('Cluster')[['AQI','PM2.5','PM10']].mean().round(2)
cluster_labels  = cluster_summary['AQI'].rank().map(
    {1.0:'Low Pollution', 2.0:'Medium Pollution', 3.0:'High Pollution'})
cluster_summary['Interpretation'] = cluster_labels

print('=== K-Means Cluster Summary ===')
print(cluster_summary)

In [ ]:
# Cluster bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors_cl = ['#2ecc71','#f1c40f','#e74c3c']
ax.bar([f'Cluster {i}' for i in cluster_summary.index],
       cluster_summary['AQI'], color=colors_cl, edgecolor='white')
ax.set_title('Average AQI per K-Means Cluster', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Average AQI', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/charts/chart7_kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nConclusion: Yes — the three clusters represent meaningful pollution groups'
      ' (low, medium, high) based on AQI and PM2.5 values.')

---
## Part H: PCA Visualization

In [ ]:
# Apply PCA — reduce to 2 components
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cl_sc)

var1      = round(pca.explained_variance_ratio_[0] * 100, 1)
var2      = round(pca.explained_variance_ratio_[1] * 100, 1)
total_var = round(var1 + var2, 1)

print(f'PC1 Variance Explained: {var1}%')
print(f'PC2 Variance Explained: {var2}%')
print(f'Total Variance Explained: {total_var}%')

# PCA scatter plot colored by AQI_Category
fig, ax = plt.subplots(figsize=(9, 6))
for cat, color in zip(cat_order, cat_colors):
    mask = df['AQI_Category'] == cat
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.5, s=20, label=cat, color=color)
ax.set_title(f'PCA Scatter Plot — Colored by AQI Category\n(PC1={var1}%, PC2={var2}%)',
             fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel(f'PC1 ({var1}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({var2}% variance)', fontsize=11)
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('outputs/charts/chart8_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConclusion: Yes — PCA captured {total_var}% of total variance in 2 components.'
      ' Clear category separation is visible along PC1 (pollution intensity axis).')

---
## Section 9: Final Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method'  : ['KNN', 'Naive Bayes', 'K-Means', 'PCA'],
    'Type'    : ['Supervised','Supervised','Unsupervised','Dimensionality Reduction'],
    'Purpose' : ['Predict AQI Category','Predict AQI Category',
                 'Group similar AQ records','Visualize data in 2D'],
    'Result'  : [f'Accuracy = {knn_results[best_k]}% (k={best_k})',
                 f'Accuracy = {nb_acc}%',
                 'Number of clusters = 3',
                 f'Variance explained = {total_var}%']
})
print(comparison.to_string(index=False))

---
## Conclusion

This project successfully implemented a full introductory data science pipeline on global AQI data:

- **PM2.5** is the strongest predictor of AQI (correlation > 0.95)
- **Delhi** has the highest average AQI; **New York** has the lowest
- AQI showed a **slow downward trend** from 2015–2025, suggesting gradual improvement
- **Naive Bayes** outperformed KNN in classification accuracy
- **K-Means** produced three meaningful, interpretable pollution clusters
- **PCA** confirmed that pollutant features drive most of the data's variance

---
*Note: AI tools (Claude by Anthropic) were used to assist in code generation and analysis for this assignment.*